# Лабораторна робота 6. Попередньо навчена нейронна мережа EfficientNetB0

Ноутбук 4 з 4. Виконав: Бабин Богдан Олегович, група 443-4

**EfficientNetB0** (2019, Google). Збалансоване масштабування глибини, ширини та роздільності мережі. Найменша з чотирьох — ~5,3 млн параметрів. Розмір вхідного зображення — 224×224.
Мережу навчено на наборі ImageNet (1,2 млн зображень, 1000 класів), тому навчати її не потрібно — завантажуємо готові ваги.

**Перед запуском:** Среда выполнения → T4 GPU → «Выполнить все». На комірці завантаження оберіть свої зображення.

> **Оновлення до нового API Keras 3:** `tensorflow.keras.preprocessing.image` → `keras.utils.load_img / img_to_array`.
> Для кожної мережі імпортуються її **власні** `preprocess_input` і `decode_predictions` — у різних мереж різна попередня обробка зображень.

In [ ]:
from keras.applications.efficientnet import EfficientNetB0, preprocess_input, decode_predictions
from keras.utils import load_img, img_to_array
from google.colab import files
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os, shutil, time
%matplotlib inline

## Завантажуємо попередньо навчену мережу EfficientNetB0

In [ ]:
model = EfficientNetB0(weights='imagenet')

In [ ]:
model.summary()
print(f'Параметрів: {model.count_params():,}  (~{model.count_params() * 4 / 2**20:.0f} МБ ваг)')
print('Шарів:', len(model.layers), '| Вхід:', model.input_shape)

## Розпізнавання зображення з базового прикладу

In [ ]:
!wget -q -O cat.jpg https://cdn.britannica.com/39/7139-050-A88818BB/Himalayan-chocolate-point.jpg || true

In [ ]:
if os.path.exists('cat.jpg') and os.path.getsize('cat.jpg') > 0:
    img = load_img('cat.jpg', target_size=(224, 224))
    plt.imshow(img); plt.axis('off'); plt.show()
    # Попередня обробка зображення
    x = img_to_array(img)
    x = np.expand_dims(x, axis=0)
    x = preprocess_input(x)
    # Розпізнавання
    preds = model.predict(x)
    print('Результати розпізнавання:', decode_predictions(preds, top=3)[0])
else:
    print('Зображення з інтернету не завантажилось — переходимо до власних зображень')

## Розпізнавання власних зображень

Завантажте **ті самі** 10 зображень у кожному з 4 ноутбуків, щоб порівняння мереж було чесним.
Назвіть файли з префіксом класу, як у лабі 4: `airplane_`, `automobile_`, `bird_`, `cat_`, `deer_`,
`dog_`, `frog_`, `horse_`, `ship_`, `truck_` (наприклад `cat_1.jpg`).
Тоді програма сама перевірить, чи належить відповідь мережі до потрібної групи класів ImageNet
(наприклад, `tabby` і `Persian_cat` — це «кіт», `sports_car` і `cab` — «автомобіль»).

In [ ]:
os.makedirs('my_images', exist_ok=True)
uploaded = files.upload()
for name in uploaded:
    shutil.move(name, os.path.join('my_images', name))
image_paths = sorted(os.path.join('my_images', f) for f in os.listdir('my_images'))
print(len(image_paths), 'файлів:', [os.path.basename(p) for p in image_paths])

In [ ]:
# Групи класів ImageNet, які відповідають класам CIFAR-10 (номери перевірено за imagenet_class_index.json)
imagenet_groups = {
    'airplane':   {404, 895, 908},                                    # airliner, warplane, wing
    'automobile': {407, 436, 468, 479, 511, 581, 609, 627, 656, 661, 717, 751, 817},  # cab, convertible, jeep, sports_car ...
    'bird':       set(range(7, 25)) | set(range(80, 101)) | set(range(127, 147)),
    'cat':        {281, 282, 283, 284, 285},                          # tabby, tiger_cat, Persian, Siamese, Egyptian
    'deer':       {351, 352, 353},                                    # оленя в ImageNet немає — найближчі: hartebeest, impala, gazelle
    'dog':        set(range(151, 269)),                               # 118 порід собак
    'frog':       {30, 31, 32},                                       # bullfrog, tree_frog, tailed_frog
    'horse':      {339, 603},                                         # sorrel, horse_cart
    'ship':       {403, 472, 484, 510, 554, 576, 625, 628, 724, 780, 814, 833, 871, 914},
    'truck':      {555, 569, 675, 717, 734, 864, 867},                # fire_engine, garbage_truck, trailer_truck ...
}

def true_group(path):
    name = os.path.basename(path).lower()
    for key in imagenet_groups:
        if name.startswith(key):
            return key
    return None

In [ ]:
def load_and_prepare(path, size):
    img = load_img(path, target_size=(size, size))
    x = img_to_array(img)
    x = np.expand_dims(x, axis=0)
    return preprocess_input(x)

def recognize_all(model, model_name, size, paths):
    rows, times = [], []
    cols = 5; rows_n = int(np.ceil(len(paths) / cols))
    plt.figure(figsize=(cols * 3.2, rows_n * 3.6))
    for i, p in enumerate(paths):
        x = load_and_prepare(p, size)
        t0 = time.time(); preds = model.predict(x, verbose=0); times.append(time.time() - t0)
        top5_idx = np.argsort(preds[0])[::-1][:5]
        top5 = decode_predictions(preds, top=5)[0]
        g = true_group(p)
        ok1 = None if g is None else bool(top5_idx[0] in imagenet_groups[g])
        ok5 = None if g is None else bool(any(k in imagenet_groups[g] for k in top5_idx))
        plt.subplot(rows_n, cols, i + 1)
        plt.imshow(load_img(p)); plt.axis('off')
        plt.title(f'{top5[0][1]}\n{top5[0][2]*100:.1f}%', fontsize=10,
                  color='black' if ok1 is None else ('green' if ok1 else 'red'))
        rows.append({'модель': model_name, 'файл': os.path.basename(p), 'клас (з назви)': g or '?',
                     'топ-1': top5[0][1], 'впевненість, %': round(top5[0][2] * 100, 1),
                     'топ-5': ', '.join(f'{lbl} {pr*100:.0f}%' for _, lbl, pr in top5),
                     'вірно топ-1': ok1, 'вірно топ-5': ok5})
    plt.suptitle(model_name); plt.tight_layout(); plt.show()
    df = pd.DataFrame(rows)
    known = df[df['вірно топ-1'].notna()]
    summary = {'модель': model_name,
               'точність топ-1, %': round(float(known['вірно топ-1'].mean()) * 100, 1) if len(known) else None,
               'точність топ-5, %': round(float(known['вірно топ-5'].mean()) * 100, 1) if len(known) else None,
               'середня впевненість, %': round(float(df['впевненість, %'].mean()), 1),
               'параметрів, млн': round(model.count_params() / 1e6, 1),
               'розмір ваг, МБ': round(model.count_params() * 4 / 2**20),
               'час на 1 фото, мс': round(float(np.mean(times[1:] if len(times) > 1 else times)) * 1000)}
    return df, summary

In [ ]:
df, summary = recognize_all(model, 'EfficientNetB0', 224, image_paths)
pd.set_option('display.max_colwidth', None)
df

### Підсумок для мережі

In [ ]:
print(summary)
df.to_csv('lab6_EfficientNetB0.csv', index=False)
pd.DataFrame([summary]).to_csv('lab6_EfficientNetB0_summary.csv', index=False)
files.download('lab6_EfficientNetB0.csv')
pd.DataFrame([summary])

## Порівняння всіх чотирьох мереж на однакових зображеннях

Завантажуємо по черзі VGG16, ResNet50, InceptionV3 і EfficientNetB0 та розпізнаємо ті самі зображення.
Кожна мережа використовує свої `preprocess_input` і `decode_predictions`.

In [ ]:
import importlib, gc
import keras.applications as KA

all_rows, all_summaries = [], []
for cls, mod, size in [('VGG16', 'vgg16', 224), ('ResNet50', 'resnet50', 224),
                       ('InceptionV3', 'inception_v3', 299), ('EfficientNetB0', 'efficientnet', 224)]:
    m = getattr(KA, mod)
    preprocess_input, decode_predictions = m.preprocess_input, m.decode_predictions   # свої для кожної мережі
    net = getattr(m, cls)(weights='imagenet')
    d, s = recognize_all(net, cls, size, image_paths)
    all_rows.append(d); all_summaries.append(s)
    del net; gc.collect()

comparison = pd.DataFrame(all_summaries)
comparison.to_csv('lab6_comparison.csv', index=False)
comparison

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
comparison.plot.bar(x='модель', y=['точність топ-1, %', 'точність топ-5, %'], ax=ax[0], rot=0, color=['#4c72b0', '#9db4d6'])
ax[0].set_title('Точність на власних зображеннях'); ax[0].set_ylim(0, 105); ax[0].set_xlabel('')
comparison.plot.bar(x='модель', y='параметрів, млн', ax=ax[1], rot=0, legend=False, color='#dd8452')
ax[1].set_title('Кількість параметрів, млн'); ax[1].set_xlabel('')
comparison.plot.bar(x='модель', y='середня впевненість, %', ax=ax[2], rot=0, legend=False, color='#55a868')
ax[2].set_title('Середня впевненість топ-1, %'); ax[2].set_xlabel(''); ax[2].set_ylim(0, 105)
for a_ in ax:
    for c in a_.containers: a_.bar_label(c, fmt='%.0f', fontsize=8)
plt.tight_layout(); plt.show()

### Відповіді всіх мереж по кожному зображенню

In [ ]:
answers = pd.concat(all_rows).pivot(index='файл', columns='модель', values='топ-1')
answers = answers[['VGG16', 'ResNet50', 'InceptionV3', 'EfficientNetB0']]
answers.to_csv('lab6_answers.csv')
answers

In [ ]:
!zip -q -r lab6_images.zip my_images
for f in ['lab6_comparison.csv', 'lab6_answers.csv', 'lab6_images.zip']:
    files.download(f)

## Висновки (EfficientNetB0)

*(заповніть після запуску)*

- Вірно розпізнано: топ-1 ... з 10, топ-5 ... з 10.
- На яких зображеннях помилки і чому: ...
- Розмір мережі та швидкість: ...